# Feature Engineering for supply-anomaly-classification

This notebook builds a compact and interpretable feature set for order-level anomaly detection using the available transactional data.

### Main goals
- Derive operational and financial signals from the raw order table
- Capture shipping, pricing, discount, and calendar deviations
- Keep the feature set interpretable and leakage-safe
- Prepare the dataset for downstream anomaly modeling

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

from pathlib import Path
from config.settings import EXPORT_DIR, DATA_DIR

In [2]:
export_dir = Path(EXPORT_DIR)
data_dir = Path(DATA_DIR)

---
## Load prepared datasets

In [3]:
# Load dataco
df = pd.read_parquet(export_dir / 'dataco_df.parquet')

In [4]:
print('df', df.shape)
df.head()

df (171962, 13)


,Days for shipping (real),Days for shipment (scheduled),Department Name,Market,order date (DateOrders),Order Item Discount,Order Item Product Price,Order Item Quantity,Order Item Total,Order Status,Product Card Id,Product Category Id,Shipping Mode
0,2,4,Fan Shop,LATAM,2015-01-01 00:00:00,60.0,299.980011,1,239.980011,COMPLETE,957,43,Standard Class
1,3,4,Fan Shop,LATAM,2015-01-01 00:21:00,6.0,199.990005,1,193.990005,PENDING_PAYMENT,1073,48,Standard Class
2,3,4,Apparel,LATAM,2015-01-01 00:21:00,22.1,129.990005,1,107.890007,PENDING_PAYMENT,403,18,Standard Class
3,3,4,Golf,LATAM,2015-01-01 00:21:00,22.5,50.000000,5,137.500000,PENDING_PAYMENT,502,24,Standard Class
4,5,4,Apparel,LATAM,2015-01-01 01:03:00,3.0,59.990002,5,284.950012,COMPLETE,365,17,Standard Class


---
## Shipping features

Shipping-related features capture delivery delays, service level differences, and coarse shipping-speed categories.

In [5]:
df['shipping_delay'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']

In [6]:
shipping_stats = (
    df.groupby(
        ['Shipping Mode', 'Market'], observed=True
        )['shipping_delay']
    .quantile([0.25, 0.50, 0.75])
    .unstack()
    .rename(columns={
        0.25: 'temp_shipping_q1',
        0.50: 'shipping_median',
        0.75: 'temp_shipping_q3'})
)

shipping_stats['temp_shipping_iqr'] = shipping_stats['temp_shipping_q3'] - shipping_stats['temp_shipping_q1']
shipping_stats.to_parquet(data_dir / 'shipping_stats.parquet')
shipping_stats

temp_shipping_q1  shipping_median  \
Shipping Mode  Market                                            
First Class    Africa                     1.0              1.0   
               Europe                     1.0              1.0   
               LATAM                      1.0              1.0   
               Pacific Asia               1.0              1.0   
               USCA                       1.0              1.0   
Same Day       Africa                     0.0              0.0   
               Europe                     0.0              0.0   
               LATAM                      0.0              1.0   
               Pacific Asia               0.0              0.0   
               USCA                       0.0              0.0   
Second Class   Africa                     1.0              2.0   
               Europe                     1.0              2.0   
               LATAM                      1.0              2.0   
               Pacific Asia               1.0              2.0   
               USCA                       1.0              2.0   
Standard Class Africa                    -1.0              0.0   
               Europe                    -1.0              0.0   
               LATAM                     -1.0              0.0   
               Pacific Asia              -1.0              0.0   
               USCA                      -1.0              0.0   

                             temp_shipping_q3  temp_shipping_iqr  
Shipping Mode  Market                                             
First Class    Africa                     1.0                0.0  
               Europe                     1.0                0.0  
               LATAM                      1.0                0.0  
               Pacific Asia               1.0                0.0  
               USCA                       1.0                0.0  
Same Day       Africa                     1.0                1.0  
               Europe                     1.0                1.0  
               LATAM                      1.0                1.0  
               Pacific Asia               1.0                1.0  
               USCA                       1.0                1.0  
Second Class   Africa                     3.0                2.0  
               Europe                     3.0                2.0  
               LATAM                      3.0                2.0  
               Pacific Asia               3.0                2.0  
               USCA                       3.0                2.0  
Standard Class Africa                     1.0                2.0  
               Europe                     1.0                2.0  
               LATAM                      1.0                2.0  
               Pacific Asia               1.0                2.0  
               USCA                       1.0                2.0

In [7]:
df = df.join(shipping_stats, on=['Shipping Mode', 'Market'])

# Shipping deviation from the group's median behavior across shipping mode and market
df['shipping_deviation'] = df['shipping_delay'] - df['shipping_median']

# Robustly scaled shipping deviation; core signal for anomaly detection
df['shipping_anomaly_raw'] = df['shipping_deviation'].abs() / (df['temp_shipping_iqr'] + 1e-5)

---
## Discount variance features

These features measure how far each order deviates from its local price and discount benchmark.

In [8]:
df['discount_rate'] = df['Order Item Discount'] / (df['Order Item Product Price'] * df['Order Item Quantity'])

In [9]:
discount_stats = (
    df.groupby(
        'Product Card Id', observed=True
        )['discount_rate']
    .quantile([0.25, 0.50, 0.75])
    .unstack()
    .rename(columns={
        0.25: 'temp_discount_q1',
        0.50: 'discount_median',
        0.75: 'temp_discount_q3'})
)

discount_stats['temp_discount_iqr'] = discount_stats['temp_discount_q3'] - discount_stats['temp_discount_q1']
discount_stats.to_parquet(data_dir / 'discount_stats.parquet')
discount_stats

,temp_discount_q1,discount_median,temp_discount_q3,temp_discount_iqr
Product Card Id,,,,
19,0.037503,0.080006,0.150012,0.112509
24,0.040005,0.090011,0.150019,0.110014
35,0.040002,0.090006,0.150009,0.110007
37,0.040011,0.090026,0.149971,0.109960
44,0.050008,0.100017,0.166694,0.116686
...,...,...,...,...
982,0.047503,0.110007,0.170011,0.122508
1004,0.040002,0.100005,0.160008,0.120006
1014,0.040016,0.090036,0.159997,0.119981


In [10]:
df = df.join(discount_stats, on='Product Card Id')

# Discount rate deviation from the product's median historical discount
df['discount_deviation'] = df['discount_rate'] - df['discount_median']

# Robustly scaled discount deviation; captures pricing anomalies at order level
df['discount_anomaly_raw'] = df['discount_deviation'].abs() / (df['temp_discount_iqr'] + 1e-5)

In [11]:
remove_cols = df.columns[df.columns.str.startswith('temp_')]
df.drop(columns=remove_cols, inplace=True)

---
## Feature audit
A final audit helps verify data types, memory usage, and the overall shape of the engineered dataset.

In [12]:
feature_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_pct': df.isna().mean() * 100,
    'n_unique': df.nunique()
}).sort_values('missing_pct', ascending=False)

feature_summary

,dtype,missing_pct,n_unique
Days for shipping (real),int8,0.0,7
Days for shipment (scheduled),int8,0.0,4
Department Name,category,0.0,6
Market,category,0.0,5
order date (DateOrders),datetime64[ns],0.0,57349
Order Item Discount,float32,0.0,725
Order Item Product Price,float32,0.0,57
Order Item Quantity,int8,0.0,5
Order Item Total,float32,0.0,2309
Order Status,category,0.0,6


In [13]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Days for shipping (real),171962.0,3.497249,0.0,2.0,3.0,5.0,6.0,1.623859
Days for shipment (scheduled),171962.0,2.931944,0.0,2.0,4.0,4.0,4.0,1.37446
order date (DateOrders),171962,2016-05-17 01:52:21.930310400,2015-01-01 00:00:00,2015-09-08 21:48:15,2016-05-17 09:27:00,2017-01-23 09:16:00,2017-09-30 23:59:00,NaN
Order Item Discount,171962.0,19.914879,0.0,5.76,14.0,29.99,500.0,19.223717
Order Item Product Price,171962.0,133.594696,9.99,50.0,59.990002,199.990005,1999.98999,117.608238
Order Item Quantity,171962.0,2.182383,1.0,1.0,1.0,3.0,5.0,1.46642
Order Item Total,171962.0,153.802795,9.99,75.0,123.490005,199.950012,1699.98999,101.833534
Product Card Id,171962.0,660.565799,19.0,403.0,627.0,1004.0,1073.0,310.473162
Product Category Id,171962.0,30.099208,2.0,18.0,29.0,45.0,48.0,13.716604
shipping_delay,171962.0,0.565305,-2.0,0.0,1.0,1.0,4.0,1.490738


In [14]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171962 entries, 0 to 171961
Data columns (total 21 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Days for shipping (real)       171962 non-null  int8          
 1   Days for shipment (scheduled)  171962 non-null  int8          
 2   Department Name                171962 non-null  category      
 3   Market                         171962 non-null  category      
 4   order date (DateOrders)        171962 non-null  datetime64[ns]
 5   Order Item Discount            171962 non-null  float32       
 6   Order Item Product Price       171962 non-null  float32       
 7   Order Item Quantity            171962 non-null  int8          
 8   Order Item Total               171962 non-null  float32       
 9   Order Status                   171962 non-null  category      
 10  Product Card Id                171962 non-null  int16         
 11  

In [15]:
float_cols = ['shipping_median', 'shipping_anomaly_raw', 'discount_rate', 
              'discount_median', 'discount_deviation', 'discount_anomaly_raw']
int_cols = ['shipping_deviation']

df[float_cols] = df[float_cols].astype('float32')
df[int_cols] = df[int_cols].astype('int8')

In [16]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Days for shipping (real),171962.0,3.497249,0.0,2.0,3.0,5.0,6.0,1.623859
Days for shipment (scheduled),171962.0,2.931944,0.0,2.0,4.0,4.0,4.0,1.37446
order date (DateOrders),171962,2016-05-17 01:52:21.930310400,2015-01-01 00:00:00,2015-09-08 21:48:15,2016-05-17 09:27:00,2017-01-23 09:16:00,2017-09-30 23:59:00,NaN
Order Item Discount,171962.0,19.914879,0.0,5.76,14.0,29.99,500.0,19.223717
Order Item Product Price,171962.0,133.594696,9.99,50.0,59.990002,199.990005,1999.98999,117.608238
Order Item Quantity,171962.0,2.182383,1.0,1.0,1.0,3.0,5.0,1.46642
Order Item Total,171962.0,153.802795,9.99,75.0,123.490005,199.950012,1699.98999,101.833534
Product Card Id,171962.0,660.565799,19.0,403.0,627.0,1004.0,1073.0,310.473162
Product Category Id,171962.0,30.099208,2.0,18.0,29.0,45.0,48.0,13.716604
shipping_delay,171962.0,0.565305,-2.0,0.0,1.0,1.0,4.0,1.490738


In [17]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171962 entries, 0 to 171961
Data columns (total 21 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Days for shipping (real)       171962 non-null  int8          
 1   Days for shipment (scheduled)  171962 non-null  int8          
 2   Department Name                171962 non-null  category      
 3   Market                         171962 non-null  category      
 4   order date (DateOrders)        171962 non-null  datetime64[ns]
 5   Order Item Discount            171962 non-null  float32       
 6   Order Item Product Price       171962 non-null  float32       
 7   Order Item Quantity            171962 non-null  int8          
 8   Order Item Total               171962 non-null  float32       
 9   Order Status                   171962 non-null  category      
 10  Product Card Id                171962 non-null  int16         
 11  

In [18]:
# Quick correlation check against the secondary target
df.corr(numeric_only=True)['shipping_anomaly_raw'].sort_values(ascending=False)

shipping_anomaly_raw             1.000000
Days for shipment (scheduled)    0.351725
Days for shipping (real)         0.247137
shipping_deviation               0.008818
discount_deviation               0.003917
discount_rate                    0.003893
Order Item Discount              0.001326
Product Category Id              0.001089
Product Card Id                  0.001010
Order Item Product Price         0.000441
discount_median                 -0.000233
discount_anomaly_raw            -0.000340
Order Item Total                -0.000753
Order Item Quantity             -0.000932
shipping_delay                  -0.055084
shipping_median                 -0.116985
Name: shipping_anomaly_raw, dtype: float64

In [19]:
# Quick correlation check against the secondary target
df.corr(numeric_only=True)['discount_anomaly_raw'].sort_values(ascending=False)

discount_anomaly_raw             1.000000
discount_rate                    0.396290
discount_deviation               0.395807
Order Item Discount              0.163268
Product Card Id                  0.009918
Product Category Id              0.009382
discount_median                  0.008484
shipping_median                  0.001059
shipping_delay                   0.000702
Days for shipping (real)         0.000461
shipping_deviation               0.000159
Days for shipment (scheduled)   -0.000217
shipping_anomaly_raw            -0.000340
Order Item Product Price        -0.007929
Order Item Quantity             -0.067534
Order Item Total                -0.093982
Name: discount_anomaly_raw, dtype: float64

In [20]:
# Export feature-engineered dataset
df.to_parquet(data_dir / 'features_df.parquet')
print('Feature engineered dataset saved.')

Feature engineered dataset saved.


## Key feature engineering takeaways

- Shipping delay is one of the clearest operational anomaly signals
- Price and discount variances are more informative than raw values alone